# Gaze Attention Zone Classifier

Trains a **Random Forest** classifier on synthetic gaze features to predict three attention zones:

| Zone | Meaning |
|---|---|
| `on_screen` | Subject looking at the display |
| `peripheral` | Gaze near screen edge or slightly off-screen |
| `away` | Clearly looking away |

**Features:** `gaze_ratio_h`, `gaze_ratio_v`, `yaw`, `dir_h`, `dir_v`

Sections:
1. Generate & explore synthetic training data
2. Train / test split — Random Forest baseline
3. 5-fold cross-validation comparison (RF vs SVM vs MLP)
4. Confusion matrix
5. Feature importances
6. Save model
7. Apply to a real session CSV (optional)


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, confusion_matrix
)
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sys.path.insert(0, str(Path('../src')))
from gaze_classifier import FEATURES, ZONES, GazeZoneClassifier, generate_training_data

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')


## 1. Generate Synthetic Training Data

`generate_training_data()` produces realistic distributions for each zone grounded
in gaze-research norms. The distributions overlap — as they would in real data —
so the classifier must learn to combine multiple cues.


In [ ]:
X, y = generate_training_data(n_per_class=600, seed=42)
df = pd.DataFrame(X, columns=list(FEATURES))
df['zone'] = y

print(f'Total samples : {len(X)}')
print(f'Class balance :')
print(df['zone'].value_counts().to_string())
df.groupby('zone')[list(FEATURES)].mean().round(3)


### Feature distributions by zone

In [ ]:
COLORS = {'on_screen': '#4CAF50', 'peripheral': '#FF9800', 'away': '#F44336'}

fig, axes = plt.subplots(1, len(FEATURES), figsize=(18, 4))
for ax, feat in zip(axes, FEATURES):
    for zone in ZONES:
        vals = df.loc[df['zone'] == zone, feat]
        ax.hist(vals, bins=30, alpha=0.55, label=zone, color=COLORS[zone])
    ax.set_title(feat)
    ax.set_xlabel('value')
    if feat == FEATURES[0]:
        ax.set_ylabel('count')
        ax.legend(fontsize=8)

plt.suptitle('Feature distributions by attention zone', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


### 2D scatter — iris ratio space vs fused direction space

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for zone in ZONES:
    m = df['zone'] == zone
    axes[0].scatter(
        df.loc[m, 'gaze_ratio_h'], df.loc[m, 'gaze_ratio_v'],
        s=8, alpha=0.4, label=zone, color=COLORS[zone]
    )
    axes[1].scatter(
        df.loc[m, 'dir_h'], df.loc[m, 'dir_v'],
        s=8, alpha=0.4, label=zone, color=COLORS[zone]
    )

for ax, title in zip(axes, ['Iris ratio space', 'Fused gaze direction space']):
    ax.set_title(title)
    ax.legend(markerscale=2)

axes[0].set_xlabel('gaze_ratio_h'); axes[0].set_ylabel('gaze_ratio_v')
axes[1].set_xlabel('dir_h');        axes[1].set_ylabel('dir_v')

plt.tight_layout()
plt.show()


## 2. Train / Test Split — Random Forest Baseline

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_tr)}  |  Test: {len(X_te)}')

clf = GazeZoneClassifier(n_estimators=100, random_state=42)
clf.train(X_tr, y_tr)

y_pred = [clf.predict(x) for x in X_te]
print()
print(classification_report(y_te, y_pred, target_names=list(ZONES)))


## 3. 5-Fold Cross-Validation Comparison

Comparing three model families to contextualise the Random Forest performance.
All three use a StandardScaler preprocessing step.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Random Forest': Pipeline([
        ('sc', StandardScaler()),
        ('m',  RandomForestClassifier(100, random_state=42, class_weight='balanced')),
    ]),
    'SVM (RBF)': Pipeline([
        ('sc', StandardScaler()),
        ('m',  SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=42)),
    ]),
    'MLP (100,50)': Pipeline([
        ('sc', StandardScaler()),
        ('m',  MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=400, random_state=42)),
    ]),
}

cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    cv_scores[name] = scores
    print(f'{name:20s}  mean={scores.mean():.4f}  std={scores.std():.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

names = list(cv_scores)
means = [cv_scores[n].mean() for n in names]
stds  = [cv_scores[n].std()  for n in names]
bars  = ax.bar(
    names, means, yerr=stds, capsize=6,
    color=['#4CAF50', '#2196F3', '#9C27B0'], alpha=0.8,
    error_kw={'elinewidth': 1.5}
)
ax.set_ylim(0.7, 1.02)
ax.set_ylabel('Accuracy (5-fold CV)')
ax.set_title('Model comparison — 5-fold stratified cross-validation')
for bar, m in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
        f'{m:.3f}', ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.show()


## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_te, y_pred, labels=list(ZONES))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=list(ZONES)).plot(
    ax=ax, colorbar=True, cmap='Blues'
)
ax.set_title('Confusion Matrix — Random Forest (test set)')
plt.tight_layout()
plt.show()


## 5. Feature Importances

Mean decrease in impurity across all 100 trees — higher values indicate
more discriminative features. The fused direction signals (`dir_h`, `dir_v`)
typically dominate because they combine both iris position and head rotation.


In [ ]:
imps = clf.feature_importances()
sorted_imps = sorted(imps.items(), key=lambda kv: kv[1], reverse=True)
feat_names = [k for k, _ in sorted_imps]
feat_vals  = [v for _, v in sorted_imps]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feat_names[::-1], feat_vals[::-1], color='steelblue', alpha=0.8)
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Feature Importances — Random Forest')
for bar, val in zip(bars, feat_vals[::-1]):
    ax.text(
        val + 0.003, bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}', va='center', fontsize=9
    )
plt.tight_layout()
plt.show()

print('Feature importances:')
for k, v in sorted_imps:
    print(f'  {k:18s}  {v:.4f}')


## 6. Save Model

In [ ]:
path = clf.save()
print(f'Model saved to: {path}')


## 7. Apply to Real Session CSV (optional)

Loads the most recent `data/gaze_*.csv`.
Requires the `dir_h` / `dir_v` columns — present in sessions recorded
with the current pipeline version.


In [ ]:
candidates = sorted(Path('../data').glob('gaze_*.csv'))

if not candidates:
    print('No session CSV found in data/ — run main.py to record a session first.')
else:
    csv_path = candidates[-1]
    print(f'Loading: {csv_path}')
    df_sess = pd.read_csv(csv_path)
    df_sess['time_s'] = df_sess['timestamp'] - df_sess['timestamp'].iloc[0]

    missing = [c for c in FEATURES if c not in df_sess.columns]
    if missing:
        print(f'Missing columns: {missing}')
        print('Re-record with the current pipeline to capture dir_h / dir_v.')
    else:
        feat_df = df_sess[list(FEATURES)].fillna(0.0)
        df_sess['predicted_zone'] = [
            clf.predict(row.values) for _, row in feat_df.iterrows()
        ]

        print('\nZone distribution:')
        print(df_sess['predicted_zone'].value_counts().to_string())

        pal = {'on_screen': '#4CAF50', 'peripheral': '#FF9800', 'away': '#F44336'}
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for zone in ZONES:
            m = df_sess['predicted_zone'] == zone
            axes[0].scatter(
                df_sess.loc[m, 'dir_h'], df_sess.loc[m, 'dir_v'],
                s=6, alpha=0.45, label=zone, color=pal.get(zone, 'grey')
            )
        axes[0].set_xlabel('dir_h')
        axes[0].set_ylabel('dir_v')
        axes[0].set_title('Session — predicted zones in direction space')
        axes[0].legend(markerscale=2)

        zone_num = df_sess['predicted_zone'].map({'on_screen': 2, 'peripheral': 1, 'away': 0})
        axes[1].fill_between(df_sess['time_s'], zone_num, step='mid', alpha=0.6, color='steelblue')
        axes[1].set_yticks([0, 1, 2])
        axes[1].set_yticklabels(['away', 'peripheral', 'on_screen'])
        axes[1].set_xlabel('Time (s)')
        axes[1].set_title('Predicted Attention Zone Over Time')

        plt.tight_layout()
        plt.show()
